# RAG Evaluation

Notebook này dùng để chạy benchmark RAG và xem kết quả theo cách gọn, dễ hiểu.

**Phiên bản:** 2026-05-21 — đã đồng bộ với `policy.md` và `faq.md` mới (LightGBM, binary search hạn mức, CIC blacklist, 9 trạng thái đơn vay, loan adjustment state machine).

## Cấu trúc 5 cell

1. **Cell markdown** — hướng dẫn này.
2. **Cell code (preflight)** — kiểm tra Qdrant collection có dữ liệu mới chưa, dataset hợp lệ chưa.
3. **Cell code (benchmark)** — chạy benchmark thật qua API/RAG/Vector DB/LLM nếu bật `RUN_BENCHMARK = True`.
4. **Cell code (input check)** — kiểm tra schema dataset + so sánh với file results để phát hiện stale.
5. **Cell code (output)** — xem điểm chính, biểu đồ và các case cần xem lại.

**Mặc định notebook không tự chạy benchmark** để tránh tốn API/LLM. Khi muốn chạy lại từ đầu, đổi `RUN_BENCHMARK = True` ở cell 3 rồi Run All.

> ⚠️ **Quan trọng:** Mỗi khi `backend/rag/knowledge/*.md` thay đổi, phải re-ingest Qdrant trước khi chạy benchmark, nếu không kết quả sẽ phản ánh nội dung lỗi thời:
> ```bash
> cd backend
> PYTHONPATH=. ../.venv/bin/python -m rag.ingest --recreate
> ```

## Step 1 — Preflight: Qdrant + Dataset

Cell tiếp theo kiểm tra nhanh:
- Qdrant có chạy không và collection `creditintel-kb` có ≥ 1 chunk không.
- File dataset benchmark có tồn tại và hợp lệ JSON không.

Nếu Qdrant trống hoặc lỗi → benchmark sẽ chỉ test guardrail/intent, không kiểm tra được retrieval thực.

In [ ]:
# Cell 2 — Preflight checks
import json
import os
from pathlib import Path
from urllib.request import urlopen, Request
from urllib.error import URLError


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "RAG_eval" / "rag_benchmark_dataset.json").exists():
            return candidate
    raise FileNotFoundError("Cannot find RAG_eval/rag_benchmark_dataset.json from current directory")


ROOT = find_repo_root()
BACKEND_DIR = ROOT / "backend"
DATASET_PATH = ROOT / "RAG_eval" / "rag_benchmark_dataset.json"
RESULT_PATH = ROOT / "RAG_eval" / "rag_benchmark_results.json"
CURRENT_PIPELINE_RESULT_PATH = ROOT / "RAG_eval" / "rag_eval_results_current_pipeline_temp0.json"
RESULT_CANDIDATES = [RESULT_PATH, CURRENT_PIPELINE_RESULT_PATH]
POLICY_PATH = BACKEND_DIR / "rag" / "knowledge" / "policy.md"
FAQ_PATH = BACKEND_DIR / "rag" / "knowledge" / "faq.md"

QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")
COLLECTION = os.environ.get("QDRANT_COLLECTION", "creditintel-kb")

print("Repo root:", ROOT)
print("Dataset :", DATASET_PATH.relative_to(ROOT))
print("Results :", RESULT_PATH.relative_to(ROOT))
print("Fallback:", CURRENT_PIPELINE_RESULT_PATH.relative_to(ROOT))

# Qdrant health
try:
    with urlopen(Request(f"{QDRANT_URL}/collections/{COLLECTION}"), timeout=3) as resp:
        body = json.loads(resp.read())
        info = body.get("result", {})
        points = info.get("points_count", 0)
        status = info.get("status", "unknown")
        print(f"Qdrant   : status={status}, points={points} in '{COLLECTION}'")
        if points == 0:
            print("  ⚠️  Collection rỗng — hãy chạy `python -m rag.ingest --recreate` trong backend/.")
except (URLError, Exception) as exc:  # noqa: BLE001
    print(f"Qdrant   : KHÔNG kết nối được ({exc}). Khởi động Qdrant container trước khi benchmark.")

# Knowledge freshness — nếu policy/faq mới hơn results, kết quả có thể stale
if RESULT_PATH.exists():
    result_mtime = RESULT_PATH.stat().st_mtime
    kb_mtime = max(POLICY_PATH.stat().st_mtime, FAQ_PATH.stat().st_mtime)
    if kb_mtime > result_mtime:
        print("⚠️  policy.md / faq.md mới hơn rag_benchmark_results.json — kết quả có thể đã lỗi thời.")
    else:
        print("Results: cập nhật so với knowledge base.")
else:
    print("Results: chưa có file kết quả — sẽ phải chạy benchmark lần đầu.")

## Step 2 — Có chạy lại benchmark không?

Cell tiếp theo là bước duy nhất có thể gọi hệ thống thật. Nếu `RUN_BENCHMARK = True`, notebook sẽ gọi backend, RAG chain, vector DB và LLM để tạo lại `RAG_eval/rag_benchmark_results.json`. Nếu chỉ muốn xem kết quả đã có, giữ `False`.

> Trước khi bật, đảm bảo backend đang chạy (`uvicorn main:app --reload`) và Qdrant đã được ingest với nội dung policy/faq mới nhất.

In [ ]:
# Cell 3 — Optional: run real RAG benchmark
# Set to True when you want to call API + RAG + Vector DB + LLM and regenerate results.
RUN_BENCHMARK = False

import subprocess
import sys

if RUN_BENCHMARK:
    env = os.environ.copy()
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
    print("Running real RAG benchmark. This may call OpenRouter/LLM and Qdrant.")
    completed = subprocess.run(
        [sys.executable, "tests_local/test_rag_benchmark.py"],
        cwd=BACKEND_DIR,
        env=env,
        text=True,
        capture_output=True,
        timeout=900,
    )
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
else:
    print("Benchmark skipped. Set RUN_BENCHMARK = True to regenerate RAG_eval/rag_benchmark_results.json.")

## Step 3 — Kiểm tra bộ câu hỏi đầu vào

Cell tiếp theo kiểm tra golden dataset: có đủ cột cần thiết không, có trùng `id` không, mỗi nhóm có bao nhiêu câu hỏi, và đối chiếu nhanh với results file để phát hiện trường hợp ID dataset đã đổi nhưng results chưa được rerun.

In [ ]:
# Cell 4 — Input check + staleness comparison
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 160)

with DATASET_PATH.open(encoding="utf-8") as f:
    dataset_df = pd.DataFrame(json.load(f))

required_columns = {"id", "group", "question", "ground_truth", "expected_source", "expected_behavior"}
missing_columns = required_columns - set(dataset_df.columns)
duplicate_ids = dataset_df[dataset_df.duplicated("id", keep=False)]

if missing_columns:
    raise ValueError(f"Dataset is missing columns: {sorted(missing_columns)}")
if not duplicate_ids.empty:
    display(duplicate_ids[["id", "group", "question"]])
    raise ValueError("Dataset contains duplicate ids")

display(Markdown(f"## Input dataset OK: {len(dataset_df)} questions"))
display(dataset_df["group"].value_counts().rename_axis("group").reset_index(name="questions"))

# Staleness — IDs có trong dataset nhưng không có trong results, và ngược lại
if RESULT_PATH.exists():
    with RESULT_PATH.open(encoding="utf-8") as f:
        results_ids = {row["id"] for row in json.load(f)}
    dataset_ids = set(dataset_df["id"])
    missing_in_results = sorted(dataset_ids - results_ids)
    extra_in_results = sorted(results_ids - dataset_ids)
    if missing_in_results:
        display(Markdown(
            "### ⚠️ Cases trong dataset nhưng KHÔNG có trong results — cần rerun:\n\n"
            + ", ".join(f"`{i}`" for i in missing_in_results)
        ))
    if extra_in_results:
        display(Markdown(
            "### ⚠️ Cases trong results nhưng đã bị xóa khỏi dataset — kết quả lỗi thời:\n\n"
            + ", ".join(f"`{i}`" for i in extra_in_results)
        ))
    if not missing_in_results and not extra_in_results:
        display(Markdown("### ✅ Dataset và results đồng bộ ID hoàn toàn."))

## Step 4 — Đọc kết quả và kết luận nhanh

Cell cuối đọc output của benchmark, tính các điểm chính và chỉ giữ những biểu đồ dễ hiểu nhất. Bảng `Cases to inspect` là danh sách nên xem trước khi sửa RAG hoặc prompt.

`CASE_ID = "POLICY-02"` được chọn làm spotlight vì policy mới đã viết lại cơ chế hạn mức vay sang **binary search động** — xem chi tiết để kiểm chứng chatbot có trả lời đúng cơ chế này không.

In [ ]:
# Cell 5 — Output evaluation
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import matplotlib.pyplot as plt

LOW_SCORE_THRESHOLD = 0.70
CASE_ID = "POLICY-02"


def safe_mean(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna()
    return float(values.mean()) if not values.empty else 0.0


def normalize_sources(value) -> str:
    if isinstance(value, list):
        return ", ".join(str(item) for item in value)
    if pd.isna(value):
        return ""
    return str(value)


def plot_bar(df: pd.DataFrame, x: str, y: str, title: str, ylabel: str, color: str):
    ax = df.plot(kind="bar", x=x, y=y, legend=False, color=color, figsize=(8, 4))
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=20)
    for container in ax.containers:
        ax.bar_label(container, padding=3)
    plt.tight_layout()
    plt.show()


def load_result_dataframe(path: Path) -> pd.DataFrame:
    with path.open(encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict):
        rows = payload.get("results", [])
        schema = "eval_runner"
        summary = payload.get("summary", {})
    elif isinstance(payload, list):
        rows = payload
        schema = "benchmark_api"
        summary = {}
    else:
        raise ValueError(f"Unsupported result payload in {path}")

    df = pd.DataFrame(rows)
    df.attrs["result_path"] = path
    df.attrs["schema"] = schema
    df.attrs["summary"] = summary
    return df


def row_error_text(row: pd.Series) -> str:
    for column in ("error", "predicted", "answer"):
        if column not in row:
            continue
        value = row[column]
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text:
            return text
    return ""


def all_cases_failed(df: pd.DataFrame) -> bool:
    if df.empty:
        return True

    failed = pd.Series(False, index=df.index)
    if "http_status" in df.columns:
        failed = failed | pd.to_numeric(df["http_status"], errors="coerce").ge(400).fillna(False)
    if "error" in df.columns:
        errors = df["error"].fillna("").astype(str).str.strip()
        failed = failed | errors.ne("")
    for column in ("predicted", "answer"):
        if column in df.columns:
            text = df[column].fillna("").astype(str).str.strip()
            failed = failed | text.str.startswith("ERROR:")

    return bool(failed.all())


def sample_error(df: pd.DataFrame) -> str:
    for _, row in df.iterrows():
        text = row_error_text(row)
        if text:
            return text.replace("\n", " ")[:350]
    return ""


def select_result_dataframe(paths: list[Path]) -> tuple[pd.DataFrame | None, list[tuple[Path, str]]]:
    skipped_failed: list[tuple[Path, str]] = []

    for path in paths:
        if not path.exists():
            continue
        df = load_result_dataframe(path)
        if all_cases_failed(df):
            skipped_failed.append((path, sample_error(df)))
            continue
        return df, skipped_failed

    return None, skipped_failed


def normalize_results_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    secondary_metric_label = "Relevance"

    if "answer" in df.columns and "predicted" not in df.columns:
        df["predicted"] = df["answer"]
    if "predicted" in df.columns and "answer" not in df.columns:
        df["answer"] = df["predicted"]
    if "relevance" not in df.columns and "context_precision" in df.columns:
        df["relevance"] = df["context_precision"]
        secondary_metric_label = "Context precision"

    if "source_ok" not in df.columns:
        if "context_precision" in df.columns:
            df["source_ok"] = pd.to_numeric(df["context_precision"], errors="coerce")
        else:
            df["source_ok"] = pd.NA

    if "expected_source" in df.columns:
        non_knowledge_sources = {"user_context", "none", ""}
        expected = df["expected_source"].fillna("").astype(str).str.strip()
        df.loc[expected.isin(non_knowledge_sources), "source_ok"] = pd.NA
    elif "group" in df.columns:
        df.loc[df["group"].isin(["guardrail", "personalized"]), "source_ok"] = pd.NA

    for column, default in (
        ("faithfulness", pd.NA),
        ("relevance", pd.NA),
        ("http_status", pd.NA),
        ("guardrail_pass", pd.NA),
        ("predicted", ""),
    ):
        if column not in df.columns:
            df[column] = default
    if "sources_returned" not in df.columns:
        df["sources_returned"] = [[] for _ in range(len(df))]
    if "overall" not in df.columns:
        faithfulness = pd.to_numeric(df["faithfulness"], errors="coerce")
        relevance = pd.to_numeric(df["relevance"], errors="coerce")
        df["overall"] = (faithfulness + relevance) / 2

    df.attrs["secondary_metric_label"] = secondary_metric_label
    return df


def build_failure_mask(df: pd.DataFrame) -> pd.Series:
    mask = pd.Series(False, index=df.index)
    mask = mask | pd.to_numeric(df["http_status"], errors="coerce").ge(400).fillna(False)
    mask = mask | pd.to_numeric(df["source_ok"], errors="coerce").eq(0).fillna(False)
    mask = mask | pd.to_numeric(df["faithfulness"], errors="coerce").lt(LOW_SCORE_THRESHOLD).fillna(False)
    mask = mask | pd.to_numeric(df["relevance"], errors="coerce").lt(LOW_SCORE_THRESHOLD).fillna(False)
    mask = mask | pd.to_numeric(df["overall"], errors="coerce").lt(LOW_SCORE_THRESHOLD).fillna(False)
    guardrail_failed = df["guardrail_pass"].eq(False).fillna(False)
    mask = mask | (df["group"].eq("guardrail") & guardrail_failed)
    return mask


result_paths = globals().get("RESULT_CANDIDATES", [RESULT_PATH, CURRENT_PIPELINE_RESULT_PATH])
selected_df, skipped_failed_results = select_result_dataframe(result_paths)

for skipped_path, error in skipped_failed_results:
    display(Markdown(
        "### Benchmark result ignored because every case failed\n\n"
        f"`{skipped_path.relative_to(ROOT)}` has no usable answers. Sample error: `{error}`. "
        "If you need this file, run the DB migration/init step that creates `chat_messages.error`, "
        "then rerun the benchmark."
    ))

if selected_df is None:
    if skipped_failed_results:
        display(Markdown(
            "## No usable benchmark results found\n"
            "All available result files failed before producing answers, so score tables are intentionally hidden."
        ))
    else:
        display(Markdown("## No benchmark results found\nRun cell 3 with `RUN_BENCHMARK = True` first."))
else:

    results_df = normalize_results_schema(selected_df)
    result_path = selected_df.attrs.get("result_path", RESULT_PATH)
    schema = selected_df.attrs.get("schema", "unknown")
    secondary_metric_label = results_df.attrs.get("secondary_metric_label", "Relevance")

    avg_faithfulness = safe_mean(results_df["faithfulness"])
    avg_relevance = safe_mean(results_df["relevance"])
    source_cases = pd.to_numeric(results_df["source_ok"], errors="coerce").dropna()
    source_recall = float(source_cases.mean()) if not source_cases.empty else 0.0
    guardrail_rows = results_df[(results_df["group"] == "guardrail") & results_df["guardrail_pass"].notna()]
    guardrail_rate = safe_mean(guardrail_rows["guardrail_pass"]) if not guardrail_rows.empty else pd.NA
    overall_score = safe_mean(results_df["overall"])

    scorecard_rows = [
        {"metric": "Faithfulness", "score_pct": round(avg_faithfulness * 100, 2)},
        {"metric": secondary_metric_label, "score_pct": round(avg_relevance * 100, 2)},
    ]
    if not source_cases.empty:
        scorecard_rows.append({"metric": "Source recall", "score_pct": round(source_recall * 100, 2)})
    if not pd.isna(guardrail_rate):
        scorecard_rows.append({"metric": "Guardrail", "score_pct": round(float(guardrail_rate) * 100, 2)})
    scorecard_rows.append({"metric": "Overall", "score_pct": round(overall_score * 100, 2)})
    scorecard = pd.DataFrame(scorecard_rows)

    group_summary = results_df.groupby("group").agg(
        cases=("id", "count"),
        faithfulness=("faithfulness", safe_mean),
        relevance=("relevance", safe_mean),
        overall=("overall", safe_mean),
        source_recall=("source_ok", safe_mean),
        source_cases=("source_ok", "count"),
    ).reset_index()
    group_summary.loc[group_summary["source_cases"] == 0, "source_recall"] = pd.NA
    group_summary["quality_pct"] = (group_summary["overall"] * 100).round(2)
    group_summary["source_recall_display"] = group_summary["source_recall"].apply(lambda value: "N/A" if pd.isna(value) else f"{value * 100:.1f}%")

    failures_df = results_df[build_failure_mask(results_df)].copy()
    failures_df["sources"] = failures_df["sources_returned"].apply(normalize_sources)
    failure_columns = ["id", "group", "question", "faithfulness", "relevance", "source_ok", "overall", "sources"]
    display_failures = failures_df[failure_columns].rename(columns={"relevance": secondary_metric_label})

    display(Markdown(
        f"## RAG result summary: {len(results_df)} answers\n\n"
        f"Using `{result_path.relative_to(ROOT)}` (`{schema}`)."
    ))
    display(scorecard)
    plot_bar(scorecard, "metric", "score_pct", "Main RAG scores", "Percent", "#2563eb")

    display(Markdown("### Quality by question group"))
    display(group_summary[["group", "cases", "quality_pct", "source_recall_display"]])
    plot_bar(group_summary, "group", "quality_pct", "Answer quality by group", "Percent", "#059669")

    display(Markdown(f"### Cases to inspect: {len(failures_df)}"))
    display(display_failures)

    matched = results_df[results_df["id"] == CASE_ID]
    if not matched.empty:
        row = matched.iloc[0]
        display(Markdown(f"### Detail for `{CASE_ID}`"))
        display(Markdown(f"**Question**\n\n{row['question']}"))
        display(Markdown(f"**Ground truth**\n\n{row['ground_truth']}"))
        display(Markdown(f"**Predicted**\n\n{row['predicted']}"))
    else:
        display(Markdown(f"### Case `{CASE_ID}` chưa có trong results — cần rerun benchmark."))